<a href="https://colab.research.google.com/github/uin33273/GoogleColab_practice/blob/main/%E7%AE%97%E5%AE%9A%E5%9F%BA%E6%BA%962_%EF%BC%91%E3%81%A4%E3%81%AE%E3%82%B7%E3%83%BC%E3%83%88%E3%81%A8spreadsheet%E9%80%A3%E7%B5%90.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import openpyxl
from openpyxl.utils import get_column_letter
from google.colab import files
import io
from ipywidgets import widgets
from IPython.display import display, clear_output
import re

# --- 1. ファイルアップロード ---
print("【手順】")
print("1. まず『算定区分・加算等集計表』のスプレッドシートを「Excel(.xlsx)」としてダウンロードしてください。")
print("2. 次に、以下からそれぞれのファイルをアップロードしてください。")

print("\n--- 1つ目のファイル ---")
# 修正箇所: ガイダンスのファイル名を変更
print("「1-複数を1ファイルに.csv」（B列に店舗名があるファイル）をアップロード")
up_csv = files.upload()
csv_filename = list(up_csv.keys())[0]
df_csv = pd.read_csv(io.BytesIO(up_csv[csv_filename]), encoding='cp932')

print("\n--- 2つ目のファイル ---")
print("「算定区分・加算等集計表.xlsx」（'集計'シートのD列に店舗名があるファイル）をアップロード")
up_xlsx = files.upload()
xlsx_filename = list(up_xlsx.keys())[0]
df_xlsx = pd.read_excel(io.BytesIO(up_xlsx[xlsx_filename]), sheet_name='集計')

# --- 2. 準備 ---
csv_shops = df_csv.iloc[:, 1].astype(str)
xlsx_master = df_xlsx.iloc[:, 3].dropna().astype(str).unique().tolist()

while len(df_csv.columns) < 11:
    df_csv[f'Extra_{len(df_csv.columns)}'] = ""
k_idx = 10

EXCLUDE_WORDS = ["店", "店舗", "支店"]

def get_clean_place(text, is_csv=False):
    target = text.split('_')[0] if is_csv and '_' in text else text
    clean = re.sub(r'[0-9a-zA-Z!-~ 　\(\)（）]+', '', target)
    clean = clean.replace("メソッド", "")
    for word in EXCLUDE_WORDS:
        clean = clean.replace(word, "")
    return clean

def find_matches(csv_raw, master_list):
    target_place = get_clean_place(csv_raw, is_csv=True)
    if not target_place: return []
    base_matches = [m for m in master_list if target_place == get_clean_place(m)]

    is_p_case = "児" in csv_raw or "P" in csv_raw.upper()
    is_m_case = "放" in csv_raw or "メソッド" in csv_raw or (not is_p_case)

    if is_p_case:
        p_matches = [m for m in base_matches if "P" in m.upper()]
        return p_matches if p_matches else base_matches
    if is_m_case:
        m_matches = [m for m in base_matches if "M" in m.upper()]
        return m_matches if m_matches else base_matches
    return base_matches

# --- 3. 一括照合 ---
pending_list = []
output_filename = "2-spreadsheetと複数から１つへ.csv"
print("\n照合中...")

for idx in range(len(df_csv)):
    raw_name = csv_shops[idx]
    matches = find_matches(raw_name, xlsx_master)
    if len(matches) == 1:
        df_csv.iloc[idx, k_idx] = matches[0]
    elif len(matches) > 1:
        pending_list.append({'idx': idx, 'raw': raw_name, 'matches': matches})

# --- 4. 結果出力とUI ---
def on_save_clicked(b):
    df_csv.to_csv(output_filename, index=False, encoding='cp932')
    files.download(output_filename)
    print(f"\n【{output_filename}】のダウンロードを開始しました。")

save_btn = widgets.Button(description="保存してダウンロード", button_style='success', layout={'width': '250px'})
save_btn.on_click(on_save_clicked)

if pending_list:
    print(f"\n候補が複数ある行が {len(pending_list)} 件あります。選択してください。")
    for item in pending_list:
        idx = item['idx']
        dropdown = widgets.Dropdown(options=["(スキップ)"] + item['matches'],
                                   description=f"行{idx+1}:",
                                   style={'description_width': 'initial'})

        def make_update(row_idx):
            def _update(change):
                if change['new'] != "(スキップ)":
                    df_csv.iloc[row_idx, k_idx] = change['new']
            return _update

        dropdown.observe(make_update(idx), names='value')
        print(f"CSV店名: {item['raw']}")
        display(dropdown)
else:
    print("\nすべて自動判別しました。")

print("\n最後に以下のボタンを押して完了してください。")
display(save_btn)